# Bài 11 · Xử lý dữ liệu phi cấu trúc bằng LLM

**Lập trình xử lý dữ liệu · 2627-1 · Viện TTNT, UET-VNU**

Chọn **Save a copy in Drive** trước khi sửa.

## Mục tiêu bài học

1. Đổi văn bản tự do thành bảng nhãn để phân tích bằng pandas.
2. Xây dựng phương pháp đối chứng bằng từ khoá và chỉ ra hạn chế của nó.
3. Gọi LLM (large language model, mô hình ngôn ngữ lớn) qua Gemini API, yêu cầu trả lời theo schema
   (danh sách trường, kiểu và giá trị được phép); tôn trọng hạn mức gọi và dùng bộ nhớ đệm (cache).
4. Đo chất lượng của LLM và phương pháp đối chứng trên bộ nhãn tay; viết quy tắc hậu kiểm; ước tính chi phí
   bằng token (đơn vị văn bản mô hình xử lý).

Dữ liệu thật là bảng đánh giá của Santiago, mốc chụp **29/06/2026**. Phần LLM dùng 24 đánh giá minh hoạ
do người soạn bài viết; kết quả gọi `gemini-3.5-flash-lite` cho các đánh giá này (22/10/2026) đã được lưu sẵn,
nên **không có khoá API vẫn chạy hết notebook**.
Các ví dụ được kiểm tra bằng pandas **3.0.3**, google-genai **2.19** và tài liệu Gemini API ngày 21/10/2026.

## Chuẩn bị

Ô cài đặt dành cho Colab. Nếu phiên Colab đã nạp pandas phiên bản khác, chọn
**Restart session and run all** sau khi cài.
Chạy các ô demo theo thứ tự; ô **Bài tập** để bạn tự viết lời giải và nhận xét.
Ô dưới dùng file đã tải nếu có; nếu chưa có thì đọc đúng mốc chụp từ nguồn (khoảng 59 MB).

In [ ]:
%pip -q install pandas==3.0.3 "google-genai>=2.19"

In [ ]:
import json, os, re, time
from pathlib import Path
from typing import Literal

import numpy as np
import pandas as pd
from pydantic import BaseModel, Field, ValidationError

URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/data/reviews.csv.gz")
tep = Path("data/reviews_santiago_full_2026-06-29.csv.gz")
nguon = tep if tep.exists() else URL
print("pandas:", pd.__version__)

## 1. Dữ liệu phi cấu trúc

Văn bản tự do chưa dùng được với `groupby`.

### Dữ liệu dùng trong bài

Bảng đánh giá của Santiago, mốc chụp **29/06/2026**; mỗi hàng là một đánh giá.
Đây là bản đầy đủ `data/reviews.csv.gz`; bản rút gọn trong `visualisations/` không có cột `comments`.

In [ ]:

rv = pd.read_csv(nguon)
print(rv.shape)
print(rv.columns.tolist())

### Đọc thử cột comments

`repr()` in chuỗi kèm các ký tự đặc biệt, nên thấy được `\r` và thẻ `<br/>`.

In [ ]:
print(rv["comments"].str.slice(0, 50).head(3).to_string())

In [ ]:
print(repr(rv.loc[11, "comments"][:90]))

### Thẻ xuống dòng trong đánh giá

Inside Airbnb ghi chỗ xuống dòng trong đánh giá bằng thẻ HTML `<br/>`, đôi khi kèm ký tự `\r`. `\r` (carriage return) là ký tự về đầu dòng, thường là phần còn sót của cặp xuống dòng `\r\n` trên Windows. Thay bằng dấu cách, như ở lab Bài 7, để hai câu không dính vào nhau khi tìm từ khoá.

In [ ]:
co_br = rv["comments"].str.contains("<br/>", regex=False)
print(co_br.sum(), round(co_br.mean(), 3))

In [ ]:
rv["comments"] = rv["comments"].str.replace(
    r"\r?<br/>", " ", regex=True)
print(repr(rv.loc[11, "comments"][44:85]))

### Thiếu, rỗng và độ dài

Quy tắc kiểm tra của Bài 10 vẫn cần cho cột văn bản: ô thiếu, đánh giá không có nội dung.

In [ ]:
binh_luan = rv["comments"]
print("Thiếu:", binh_luan.isna().sum())
print("Chỉ có dấu chấm:", binh_luan.str.strip().eq(".").sum())
print("Trung vị độ dài:", binh_luan.str.len().median(), "ký tự")

### Câu hỏi cần trả lời

> Khách thường **chê điều gì**? Đánh giá được viết bằng **những ngôn ngữ nào**?

Bảng không có cột ghi khía cạnh bị chê hay ngôn ngữ. Thông tin này nằm trong câu chữ của từng đánh giá.

Với 690 nghìn đánh giá, từ khoá chạy được cả bảng; LLM chỉ chạy trên một mẫu (mục 5 giải thích vì sao).

### Ba mức cấu trúc của dữ liệu

| Có cấu trúc | Bán cấu trúc | Phi cấu trúc |
|---|---|---|
| bảng, mỗi cột một kiểu (`listings.csv`) | JSON, log, HTML (dữ liệu API ở Bài 6) | văn bản tự do, ảnh, âm thanh (cột `comments`) |

JSON có khoá để chương trình đọc từng trường. Văn bản tự do không có khoá như vậy.

### Biến văn bản thành bảng nhãn

Văn bản → trích xuất (từ khoá hoặc LLM) → bảng nhãn (cảm xúc, khía cạnh, ngôn ngữ) → pandas.

Mỗi đánh giá được đổi thành một hàng nhãn. Sau đó bạn dùng lại các công cụ pandas từ Bài 4–10.

### Bảng nhãn cần tạo

| comments | sentiment | aspects_positive | aspects_negative | language |
|---|---|---|---|---|
| “The flat was not clean at all, dust everywhere. Location was perfect though.” | mixed | location | cleanliness | en |
| “Súper cómodo y muy limpio. El anfitrión respondió al instante.” | positive | cleanliness, host | — | es |

`sentiment`: cảm xúc chung; `aspects_…`: khía cạnh được khen, bị chê; `language`: mã ngôn ngữ.

## 2. Phương pháp đối chứng bằng từ khoá

Phương pháp đối chứng (baseline) dùng regex của Bài 7 để có kết quả đầu tiên.

### Tìm khía cạnh bằng từ khoá

Mỗi khía cạnh có một mẫu regex gồm từ tiếng Anh và tiếng Tây Ban Nha.

In [ ]:
TU_KHOA = {
    "noise": r"noisy|noise|loud|ruido",
    "cleanliness": r"clean|dirty|limpio|sucio",
}
for khia_canh, mau in TU_KHOA.items():
    rv[khia_canh] = rv["comments"].str.contains(mau, case=False)
print(rv[["noise", "cleanliness"]].mean().round(3))

### Từ khoá không phân biệt khen và chê

In [ ]:
co_not_clean = rv["comments"].str.contains("not clean", case=False)
print(co_not_clean.sum())
print(rv.loc[co_not_clean, "cleanliness"].all())

### Đoán cảm xúc bằng từ điển

Đếm số từ tích cực trừ số từ tiêu cực. `np.select(điều kiện, nhãn, mặc định)`: mỗi hàng lấy nhãn của điều kiện đúng đầu tiên; không điều kiện nào đúng thì lấy mặc định. Đánh giá không chứa từ nào trong từ điển có điểm 0 và bị gán `mixed`.

In [ ]:
TOT = r"great|perfect|excellent|excelente"
XAU = r"bad|dirty|noisy|terrible|malo"
diem = (rv["comments"].str.count(TOT, flags=re.I)
        - rv["comments"].str.count(XAU, flags=re.I))
rv["cam_xuc_tu_dien"] = np.select(
    [diem > 0, diem < 0], ["positive", "negative"], "mixed")
print(rv["cam_xuc_tu_dien"].value_counts())

### Từ điển không hiểu phủ định

Cụm *not bad* thường là khen hay chê? Dự đoán trước khi chạy ô dưới.

In [ ]:
not_bad = rv["comments"].str.contains("not bad", case=False)
print(rv.loc[not_bad, "cam_xuc_tu_dien"].value_counts())

35 đánh giá có cụm *not bad*, thường là lời khen nhẹ; 17 đánh giá trong số đó bị gán `negative` vì có chữ *bad*.

### Ưu điểm và hạn chế của từ khoá

- Ưu điểm: chạy cả bảng trong vài giây, miễn phí; chạy lại ra đúng kết quả cũ; biết vì sao một đánh giá được gán nhãn.
- Hạn chế: không hiểu phủ định; mỗi ngôn ngữ phải soạn từ khoá riêng; bỏ sót cách viết khác, viết tắt, sai chính tả.

### Bài tập 1: Khía cạnh an toàn

Thêm khía cạnh `safety` với mẫu `r"safe|unsafe|seguro|insegur"`.

1. Bao nhiêu phần trăm đánh giá khớp mẫu?
2. In 5 đánh giá khớp đầu tiên. Khách khen hay chê sự an toàn?
3. Trong các đánh giá khớp câu 1, bao nhiêu phần trăm khớp mẫu chê `r"not safe|unsafe|insegur|peligros"`?

In [ ]:
# Bài tập 1: viết lời giải của bạn ở đây

### Vai trò của phương pháp đối chứng

Phương pháp đối chứng cho kết quả để so sánh. LLM tốn tiền và thời gian hơn, nên phải tốt hơn đủ nhiều mới đáng dùng. Chạy cả hai cách trên cùng một bộ đánh giá đã gán nhãn tay, rồi so điểm (mục 5).

## 3. Gọi LLM từ Python

Gửi yêu cầu, nhận phản hồi, như gọi API ở Bài 6.

### Giao diện chat và API

Giao diện chat: người gõ câu hỏi, người đọc câu trả lời; khó lặp lại cho hàng nghìn đánh giá. Gọi qua API: code gửi yêu cầu và nhận phản hồi, chạy trong vòng lặp và ghi kết quả vào bảng.

### Gemini API

Lấy khoá API ở [Google AI Studio](https://aistudio.google.com).

| Mô hình | Dùng cho | Miễn phí: lời gọi/phút, /ngày | Trả phí: USD / 1 triệu token vào, ra |
|---|---|---|---|
| `gemini-3.5-flash-lite` | chạy hàng loạt | 15 / 500 | 0,30 / 2,50 |
| `gemini-3.8-flash` | văn bản khó | 5 / 20 | 0,75 / 3,75 (đến 31/12/2026) |

Token là đơn vị văn bản mô hình xử lý; tiếng Anh khoảng 4 ký tự một token. Giá theo trang [Pricing](https://ai.google.dev/gemini-api/docs/pricing) ngày 19/10/2026. Hạn mức gói miễn phí (tháng 10/2026, một project): `gemini-3.5-flash-lite` 15 lời gọi/phút và 500 lời gọi/ngày; `gemini-3.8-flash` 5 lời gọi/phút và 20 lời gọi/ngày. Hạn mức của bạn xem trong AI Studio, mục Rate Limit.

> ⚠️ Ở gói miễn phí, nội dung gửi đi có thể được Google dùng để cải thiện sản phẩm. Chỉ gửi dữ liệu công khai, như đánh giá trên Inside Airbnb.

**Lấy khoá:** AI Studio → **Get API key**. Trong Colab: biểu tượng **🔑 Secrets** ở cột trái → **Add new secret**, tên `GEMINI_API_KEY`, dán khoá vào ô giá trị, bật **Notebook access**. Không có khoá thì notebook vẫn chạy hết: kết quả LLM đọc từ cache có sẵn.

### Giữ khoá API như mật khẩu

Không viết khoá vào code, không commit lên GitHub, không dán vào chat hay ảnh chụp màn hình.

In [ ]:
try:
    from google.colab import userdata    # Colab: 🔑 Secrets
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:                         # máy cá nhân
    API_KEY = os.environ.get("GEMINI_API_KEY")
CO_KHOA = bool(API_KEY)

### Lần gọi đầu tiên

`input` là prompt: văn bản yêu cầu gửi cho mô hình. `output_text` là chuỗi mô hình trả về. Ô gọi API chỉ chạy khi có khoá.

In [ ]:
from google import genai

MODEL = "gemini-3.5-flash-lite"
client = genai.Client(api_key=API_KEY) if CO_KHOA else None

In [ ]:

if CO_KHOA:
    tra_loi = client.interactions.create(
        model=MODEL,
        input="Trong một câu: đánh giá này khen gì, chê gì? "
              "Great location, so noisy.",
    )
    print(tra_loi.output_text)
else:
    print("Chưa có khoá API: bỏ qua lần gọi thử.")

### Câu trả lời tự do chưa thành cột

Hỏi cùng một câu ba lần, yêu cầu trả lời bằng văn xuôi. Câu chữ và ký hiệu đầu dòng đổi giữa các lần; nhãn là cụm từ tự do như “Căn hộ không sạch sẽ”. Vì vậy cần yêu cầu mô hình trả lời theo **schema cố định**.

In [ ]:
if CO_KHOA:
    HOI = ("Đánh giá sau khen gì, chê gì? Trả lời ngắn gọn.\n\n"
           "Đánh giá: The flat was not clean at all, dust everywhere. "
           "Location was perfect though.")
    for lan in range(3):
        tra_loi = client.interactions.create(model=MODEL, input=HOI)
        print(f"{lan + 1}. {tra_loi.output_text}")
else:
    print("Chưa có khoá API: bỏ qua.")

## 4. Đầu ra có cấu trúc

Schema là danh sách trường, kiểu và giá trị được phép. Mô hình trả JSON theo schema.

### 24 đánh giá minh hoạ

Phần LLM dùng 24 đánh giá ngắn do người soạn bài viết, bằng 5 ngôn ngữ, có đánh giá rỗng.

Ô dưới chứa 24 đánh giá minh hoạ. Quy mô này đủ nhỏ để bạn đọc hết và tự kiểm tra từng kết quả.

In [ ]:
REVIEWS = [
    (1, 'Great location, five minutes from the metro. The host was lovely!'),
    (2, 'The flat was not clean at all, dust everywhere. Location was perfect though.'),
    (3, 'Súper cómodo y muy limpio. El anfitrión respondió al instante.'),
    (4, 'Ruidoso hasta las 3am, imposible dormir. No lo recomiendo.'),
    (5, 'Apartamento bem localizado, mas o wifi caiu o tempo todo.'),
    (6, 'Nothing special, but it was cheap and did the job.'),
    (7, "L'appartement est charmant mais la rue est très bruyante la nuit."),
    (8, 'Amazing value for money, spotless bathroom, super quiet neighborhood.'),
    (9, 'Todo perfecto! Volveremos seguro.'),
    (10, 'The pictures are misleading — the room is tiny and smells of smoke.'),
    (11, 'Host cancelled last minute, we were stranded. Avoid!'),
    (12, 'Muito limpo, cama confortável, anfitriã atenciosa. Recomendo!'),
    (13, 'Decent place. A bit far from the center but the bus stop is close.'),
    (14, 'Perfect for a weekend getaway.'),
    (15, 'El barrio se siente inseguro de noche; el depto está bien.'),
    (16, 'Ottima posizione, appartamento pulito. Torneremo!'),
    (17, 'Quiet street, comfy bed, kind host — everything was great.'),
    (18, 'We found hair in the sheets and the kitchen was greasy.'),
    (19, 'So loud! The bar downstairs plays music till 2am.'),
    (20, 'Un poco caro para lo que ofrece, pero la ubicación es inmejorable.'),
    (21, 'The check-in instructions were confusing; host took hours to reply.'),
    (22, 'Cozy studio, fast wifi, walkable to everything.'),
    (23, '👍👍👍'),
    (24, ''),
]

Ô dưới chứa kết quả lần chạy của giảng viên cho 24 đánh giá. **Chỉ dùng khi bạn chưa có khoá API**: các ô gọi API ở mục này khi đó đọc kết quả từ đây. Chỉ cần chạy, không cần đọc từng dòng.

In [ ]:
# Kết quả lần chạy của giảng viên: gemini-3.5-flash-lite, 24 đánh giá minh hoạ, 22/10/2026,
# với PROMPT_TEMPLATE và ReviewInfo ở mục 4. Chỉ dùng khi chưa có khoá API.
# Khoá: "<mô hình>|<nội dung đánh giá>"; output: chuỗi JSON mô hình trả về; in_tok, out_tok: số token vào, ra.
KET_QUA_LUU_SAN = {
    'gemini-3.5-flash-lite|Great location, five minutes from the metro. The host was lovely!': {
        "output": '{\n  "sentiment": "positive",\n  "aspects_positive": [\n    "location",\n    "host"\n  ],\n  "aspects_negative": [],\n  "language": "en"\n}',
        "in_tok": 83, "out_tok": 50},
    'gemini-3.5-flash-lite|The flat was not clean at all, dust everywhere. Location was perfect though.': {
        "output": '{\n  "sentiment": "mixed",\n  "aspects_positive": [\n    "location"\n  ],\n  "aspects_negative": [\n    "cleanliness"\n  ],\n  "language": "en"\n}',
        "in_tok": 85, "out_tok": 54},
    'gemini-3.5-flash-lite|Súper cómodo y muy limpio. El anfitrión respondió al instante.': {
        "output": '{\n  "sentiment": "positive",\n  "aspects_positive": [\n    "cleanliness",\n    "host",\n    "amenities"\n  ],\n  "aspects_negative": [],\n  "language": "es"\n}',
        "in_tok": 84, "out_tok": 57},
    'gemini-3.5-flash-lite|Ruidoso hasta las 3am, imposible dormir. No lo recomiendo.': {
        "output": '{\n  "sentiment": "negative",\n  "aspects_positive": [],\n  "aspects_negative": [\n    "noise"\n  ],\n  "language": "es"\n}',
        "in_tok": 85, "out_tok": 45},
    'gemini-3.5-flash-lite|Apartamento bem localizado, mas o wifi caiu o tempo todo.': {
        "output": '{\n  "sentiment": "mixed",\n  "aspects_positive": [\n    "location"\n  ],\n  "aspects_negative": [\n    "amenities"\n  ],\n  "language": "pt"\n}',
        "in_tok": 82, "out_tok": 54},
    'gemini-3.5-flash-lite|Nothing special, but it was cheap and did the job.': {
        "output": '{\n  "sentiment": "mixed",\n  "aspects_positive": [\n    "value"\n  ],\n  "aspects_negative": [],\n  "language": "en"\n}',
        "in_tok": 81, "out_tok": 45},
    "gemini-3.5-flash-lite|L'appartement est charmant mais la rue est très bruyante la nuit.": {
        "output": '{\n  "sentiment": "mixed",\n  "aspects_positive": [\n    "amenities"\n  ],\n  "aspects_negative": [\n    "noise"\n  ],\n  "language": "fr"\n}',
        "in_tok": 86, "out_tok": 54},
    'gemini-3.5-flash-lite|Amazing value for money, spotless bathroom, super quiet neighborhood.': {
        "output": '{\n  "sentiment": "positive",\n  "aspects_positive": [\n    "value",\n    "cleanliness",\n    "noise"\n  ],\n  "aspects_negative": [],\n  "language": "en"\n}',
        "in_tok": 81, "out_tok": 56},
    'gemini-3.5-flash-lite|Todo perfecto! Volveremos seguro.': {
        "output": '{\n  "sentiment": "positive",\n  "aspects_positive": [],\n  "aspects_negative": [],\n  "language": "es"\n}',
        "in_tok": 77, "out_tok": 37},
    'gemini-3.5-flash-lite|The pictures are misleading — the room is tiny and smells of smoke.': {
        "output": '{\n  "sentiment": "negative",\n  "aspects_positive": [],\n  "aspects_negative": [\n    "cleanliness",\n    "amenities"\n  ],\n  "language": "en"\n}',
        "in_tok": 83, "out_tok": 52},
    'gemini-3.5-flash-lite|Host cancelled last minute, we were stranded. Avoid!': {
        "output": '{\n  "sentiment": "negative",\n  "aspects_positive": [],\n  "aspects_negative": [\n    "host"\n  ],\n  "language": "en"\n}',
        "in_tok": 80, "out_tok": 45},
    'gemini-3.5-flash-lite|Muito limpo, cama confortável, anfitriã atenciosa. Recomendo!': {
        "output": '{\n  "sentiment": "positive",\n  "aspects_positive": [\n    "cleanliness",\n    "host",\n    "amenities"\n  ],\n  "aspects_negative": [],\n  "language": "pt"\n}',
        "in_tok": 87, "out_tok": 57},
    'gemini-3.5-flash-lite|Decent place. A bit far from the center but the bus stop is close.': {
        "output": '{\n  "sentiment": "mixed",\n  "aspects_positive": ["location"],\n  "aspects_negative": ["location"],\n  "language": "en"\n}',
        "in_tok": 86, "out_tok": 41},
    'gemini-3.5-flash-lite|Perfect for a weekend getaway.': {
        "output": '{\n  "sentiment": "positive",\n  "aspects_positive": [],\n  "aspects_negative": [],\n  "language": "en"\n}',
        "in_tok": 75, "out_tok": 37},
    'gemini-3.5-flash-lite|El barrio se siente inseguro de noche; el depto está bien.': {
        "output": '{\n  "sentiment": "mixed",\n  "aspects_positive": [],\n  "aspects_negative": [\n    "location"\n  ],\n  "language": "es"\n}',
        "in_tok": 84, "out_tok": 45},
    'gemini-3.5-flash-lite|Ottima posizione, appartamento pulito. Torneremo!': {
        "output": '{\n  "sentiment": "positive",\n  "aspects_positive": [\n    "location",\n    "cleanliness"\n  ],\n  "aspects_negative": [],\n  "language": "it"\n}',
        "in_tok": 81, "out_tok": 51},
    'gemini-3.5-flash-lite|Quiet street, comfy bed, kind host — everything was great.': {
        "output": '{\n  "sentiment": "positive",\n  "aspects_positive": [\n    "location",\n    "cleanliness",\n    "host",\n    "noise",\n    "amenities"\n  ],\n  "aspects_negative": [],\n  "language": "en"\n}',
        "in_tok": 82, "out_tok": 67},
    'gemini-3.5-flash-lite|We found hair in the sheets and the kitchen was greasy.': {
        "output": '{\n  "sentiment": "negative",\n  "aspects_positive": [],\n  "aspects_negative": [\n    "cleanliness"\n  ],\n  "language": "en"\n}',
        "in_tok": 81, "out_tok": 46},
    'gemini-3.5-flash-lite|So loud! The bar downstairs plays music till 2am.': {
        "output": '{\n  "sentiment": "negative",\n  "aspects_positive": [],\n  "aspects_negative": [\n    "noise"\n  ],\n  "language": "en"\n}',
        "in_tok": 82, "out_tok": 45},
    'gemini-3.5-flash-lite|Un poco caro para lo que ofrece, pero la ubicación es inmejorable.': {
        "output": '{\n  "sentiment": "mixed",\n  "aspects_positive": [\n    "location"\n  ],\n  "aspects_negative": [\n    "value"\n  ],\n  "language": "es"\n}',
        "in_tok": 85, "out_tok": 53},
    'gemini-3.5-flash-lite|The check-in instructions were confusing; host took hours to reply.': {
        "output": '{\n  "sentiment": "negative",\n  "aspects_positive": [],\n  "aspects_negative": [\n    "host",\n    "amenities"\n  ],\n  "language": "en"\n}',
        "in_tok": 83, "out_tok": 51},
    'gemini-3.5-flash-lite|Cozy studio, fast wifi, walkable to everything.': {
        "output": '{\n  "sentiment": "positive",\n  "aspects_positive": [\n    "location",\n    "amenities"\n  ],\n  "aspects_negative": [],\n  "language": "en"\n}',
        "in_tok": 79, "out_tok": 51},
    'gemini-3.5-flash-lite|👍👍👍': {
        "output": '{\n  "sentiment": "positive",\n  "aspects_positive": [],\n  "aspects_negative": [],\n  "language": "und"\n}',
        "in_tok": 72, "out_tok": 37},
    'gemini-3.5-flash-lite|': {
        "output": '{\n  "sentiment": "mixed",\n  "aspects_positive": [],\n  "aspects_negative": [],\n  "language": "und"\n}',
        "in_tok": 70, "out_tok": 37},
}
print("Kết quả lưu sẵn:", len(KET_QUA_LUU_SAN), "đánh giá")

In [ ]:
minh_hoa = pd.DataFrame(REVIEWS, columns=["review_id", "comments"])
minh_hoa = minh_hoa.set_index("review_id")
ngan = minh_hoa["comments"].str.slice(0, 50)
print(ngan.loc[[2, 5, 7]].to_string())

### Yêu cầu mô hình điền mẫu

Thay vì văn xuôi, mô hình trả về đúng các trường đã định (schema). Kết quả đọc thẳng được thành một hàng của bảng.

### Khai báo schema bằng Pydantic

Pydantic là thư viện khai báo kiểu dữ liệu và kiểm tra dữ liệu theo kiểu đó. `Literal` liệt kê các giá trị được phép. `model_json_schema()` đổi lớp này thành JSON Schema để gửi cho API. Comment `#` không được gửi cho mô hình; `description` thì có, vì nằm trong schema.

In [ ]:
Aspect = Literal["location", "cleanliness", "host",
                 "value", "noise", "amenities"]

class ReviewInfo(BaseModel):
    sentiment: Literal["positive", "mixed", "negative"]
    aspects_positive: list[Aspect]   # khía cạnh được khen
    aspects_negative: list[Aspect]   # khía cạnh bị chê
    language: str = Field(description=(
        'Mã ISO 639-1, vd "en", "es"; '
        '"und" nếu không xác định được'))

print(list(ReviewInfo.model_json_schema()["properties"]))

### Prompt gồm hướng dẫn và dữ liệu

Phần hướng dẫn giữ nguyên cho mọi đánh giá; `{text}` được thay bằng từng đánh giá.

> ⚠️ Đánh giá là dữ liệu cần phân tích. Một đánh giá có câu kiểu “bỏ qua hướng dẫn trên” (prompt injection) vẫn có thể làm mô hình trả lời lệch.

In [ ]:
PROMPT_TEMPLATE = (
    "Bạn là công cụ trích xuất thông tin từ đánh giá về chỗ ở.\n"
    "Phân tích đánh giá sau và điền đúng schema. "
    "Chỉ gán khía cạnh THỰC SỰ được nhắc đến;\n"
    'đánh giá không có nội dung thì trả sentiment "mixed" '
    "và các danh sách rỗng.\n\n"
    "Review: {text}"
)

### Gọi API kèm schema

`response_format` gửi schema kèm lời gọi; mô hình trả chuỗi JSON đúng các trường. Hàm `goi_llm` gói lời gọi này và trả thêm số token. Chưa có khoá API thì ô gọi thử đọc kết quả lưu sẵn.

In [ ]:
def goi_llm(van_ban):
    tra_loi = client.interactions.create(
        model=MODEL,
        input=PROMPT_TEMPLATE.format(text=van_ban),
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": ReviewInfo.model_json_schema(),
        },
        store=False,             # máy chủ không lưu lời gọi
    )
    u = tra_loi.usage            # token suy luận tính như token ra
    ra = (u.total_output_tokens or 0) + (u.total_thought_tokens or 0)
    return tra_loi.output_text, u.total_input_tokens, ra

In [ ]:

van_ban = minh_hoa.loc[2, "comments"]
if CO_KHOA:
    chuoi_json, vao, ra = goi_llm(van_ban)
else:                      # chưa có khoá: đọc kết quả lưu sẵn
    luu = KET_QUA_LUU_SAN[f"{MODEL}|{van_ban}"]
    chuoi_json, vao, ra = luu["output"], luu["in_tok"], luu["out_tok"]
print(chuoi_json)

### Đọc và kiểm tra chuỗi JSON

`model_validate_json` đọc chuỗi JSON, kiểm tra từng trường theo schema rồi tạo đối tượng `ReviewInfo`.

In [ ]:
info = ReviewInfo.model_validate_json(chuoi_json)
print(info.sentiment, info.aspects_positive, info.aspects_negative)

### Đầu ra sai schema bị chặn

Nhãn ngoài danh sách và trường bị thiếu đều gây lỗi ngay, thay vì lặng lẽ thành `NaN` trong bảng. Schema là bước kiểm tra đầu tiên với đầu ra của LLM, giống quy tắc kiểm tra đầu vào ở Bài 10.

In [ ]:
chuoi_json = ('{"sentiment": "positive", '
              '"aspects_positive": ["wifi_speed"], '
              '"aspects_negative": []}')
try:
    ReviewInfo.model_validate_json(chuoi_json)
except ValidationError as loi:
    for e in loi.errors():
        print(e["type"], e["loc"])

### Bài tập 2: Schema chặn được gì?

Dự đoán chuỗi nào bị chặn và vì sao, rồi chạy để kiểm tra. Chuỗi nào qua được schema nhưng vẫn có nội dung đáng ngờ?

In [ ]:
THU = {
    "a": '{"sentiment": "neutral", "aspects_positive": [], '
         '"aspects_negative": [], "language": "en"}',
    "b": '{"sentiment": "positive", "aspects_positive": "host", '
         '"aspects_negative": [], "language": "es"}',
    "c": '{"sentiment": "positive", "aspects_positive": ["host", '
         '"host"], "aspects_negative": [], "language": "Spanish"}',
}

In [ ]:
# Bài tập 2: viết lời giải của bạn ở đây

### Chạy cho cả bộ đánh giá

Bộ nhớ đệm (cache) giữ kết quả đã có: chạy lại không gọi API lần nữa. Hạn mức (rate limit) là số lời gọi tối đa mỗi phút, mỗi ngày. Khoá cache gồm tên mô hình và nội dung đánh giá; đổi prompt thì phải xoá cache.

Hàm `trich_xuat` làm ba bước cho từng đánh giá: tra cache; chưa có thì gọi API, lưu vào cache, nghỉ 4 giây; kiểm tra bằng schema.

In [ ]:
LLM_CACHE = {} if CO_KHOA else dict(KET_QUA_LUU_SAN)   # chưa có khoá: bắt đầu từ kết quả lưu sẵn
def trich_xuat(van_ban):
    khoa_cache = f"{MODEL}|{van_ban}"     # mô hình + nội dung
    if khoa_cache not in LLM_CACHE:       # chỉ gọi khi chưa có
        try:
            chuoi_json, vao, ra = goi_llm(van_ban)
        except Exception as loi:         # vd 429: vượt hạn mức
            print("Lỗi API:", loi)
            return None
        LLM_CACHE[khoa_cache] = {"output": chuoi_json,
                                 "in_tok": vao, "out_tok": ra}
        time.sleep(4)                    # giữ dưới hạn mức
    try:
        chuoi_json = LLM_CACHE[khoa_cache]["output"]
        return ReviewInfo.model_validate_json(chuoi_json)
    except ValidationError:
        return None                      # sai schema: bỏ qua

In [ ]:
ket_qua = {ma: trich_xuat(t)
           for ma, t in minh_hoa["comments"].items()}
hop_le = {ma: r for ma, r in ket_qua.items() if r is not None}
print("Hợp lệ:", len(hop_le), "/", len(minh_hoa))

In [ ]:
Path("llm_cache.json").write_text(json.dumps(LLM_CACHE))
doc_lai = json.loads(Path("llm_cache.json").read_text())
print(doc_lai == LLM_CACHE)

### Chuyển kết quả về DataFrame

`explode` tách mỗi phần tử danh sách thành một hàng; sau đó chỉ cần `value_counts`.

In [ ]:
bang = {ma: r.model_dump() for ma, r in hop_le.items()}
nhan = pd.DataFrame.from_dict(bang, orient="index")
bi_che = nhan["aspects_negative"].explode().value_counts()
print(bi_che)

### Cảm xúc theo ngôn ngữ 📖 Tự học

Dòng `und` gồm đánh giá chỉ có emoji và đánh giá rỗng.

In [ ]:
print(pd.crosstab(nhan["language"], nhan["sentiment"]))

## 5. Đánh giá chất lượng và chi phí

Đúng schema chưa chắc đúng nội dung.

### Đọc lại vài kết quả

In [ ]:
for ma in [17, 24]:
    r = ket_qua[ma]
    print(ma, repr(minh_hoa.loc[ma, "comments"]))
    print("  ", r.sentiment, r.aspects_positive, r.language)

### Bộ nhãn gán tay

1. Rút ngẫu nhiên một mẫu đánh giá, có `random_state`; kiểm tra tỷ lệ các ngôn ngữ chính.
2. Viết hướng dẫn gán nhãn: thế nào là `mixed`, đánh giá rỗng gán gì, một khía cạnh có được nằm ở cả hai danh sách không.
3. Hai người gán độc lập; chỗ bất đồng thì thảo luận và sửa hướng dẫn.
4. Chạy cả đối chứng và LLM trên đúng mẫu này, rồi so với nhãn tay.

Bộ nhãn tay đóng vai trò như bộ dữ liệu kiểm thử (test set). Không có nó, nhận xét về chất lượng chỉ là cảm nhận.

Ô dưới chứa nhãn tay cho cả 24 đánh giá minh hoạ.

In [ ]:
# Nhãn tay cho cả 24 đánh giá minh hoạ.
# Quy ước gán: khu phố tính vào location; mỗi khía cạnh chỉ thuộc một danh sách theo ý chính;
# đánh giá không có nội dung: mixed, không có khía cạnh, ngôn ngữ "und".
GOLD = {
    1: {"sentiment": 'positive', "aspects_positive": ['location', 'host'],
         "aspects_negative": [], "language": 'en'},
    2: {"sentiment": 'mixed', "aspects_positive": ['location'],
         "aspects_negative": ['cleanliness'], "language": 'en'},
    3: {"sentiment": 'positive', "aspects_positive": ['cleanliness', 'host'],
         "aspects_negative": [], "language": 'es'},
    4: {"sentiment": 'negative', "aspects_positive": [],
         "aspects_negative": ['noise'], "language": 'es'},
    5: {"sentiment": 'mixed', "aspects_positive": ['location'],
         "aspects_negative": ['amenities'], "language": 'pt'},
    6: {"sentiment": 'mixed', "aspects_positive": ['value'],
         "aspects_negative": [], "language": 'en'},
    7: {"sentiment": 'mixed', "aspects_positive": [],
         "aspects_negative": ['noise'], "language": 'fr'},
    8: {"sentiment": 'positive', "aspects_positive": ['value', 'cleanliness', 'noise'],
         "aspects_negative": [], "language": 'en'},
    9: {"sentiment": 'positive', "aspects_positive": [],
         "aspects_negative": [], "language": 'es'},
    10: {"sentiment": 'negative', "aspects_positive": [],
         "aspects_negative": ['cleanliness'], "language": 'en'},
    11: {"sentiment": 'negative', "aspects_positive": [],
         "aspects_negative": ['host'], "language": 'en'},
    12: {"sentiment": 'positive', "aspects_positive": ['cleanliness', 'amenities', 'host'],
         "aspects_negative": [], "language": 'pt'},
    13: {"sentiment": 'mixed', "aspects_positive": [],
         "aspects_negative": ['location'], "language": 'en'},
    14: {"sentiment": 'positive', "aspects_positive": [],
         "aspects_negative": [], "language": 'en'},
    15: {"sentiment": 'mixed', "aspects_positive": [],
         "aspects_negative": ['location'], "language": 'es'},
    16: {"sentiment": 'positive', "aspects_positive": ['location', 'cleanliness'],
         "aspects_negative": [], "language": 'it'},
    17: {"sentiment": 'positive', "aspects_positive": ['noise', 'amenities', 'host'],
         "aspects_negative": [], "language": 'en'},
    18: {"sentiment": 'negative', "aspects_positive": [],
         "aspects_negative": ['cleanliness'], "language": 'en'},
    19: {"sentiment": 'negative', "aspects_positive": [],
         "aspects_negative": ['noise'], "language": 'en'},
    20: {"sentiment": 'mixed', "aspects_positive": ['location'],
         "aspects_negative": ['value'], "language": 'es'},
    21: {"sentiment": 'negative', "aspects_positive": [],
         "aspects_negative": ['host'], "language": 'en'},
    22: {"sentiment": 'positive', "aspects_positive": ['amenities', 'location'],
         "aspects_negative": [], "language": 'en'},
    23: {"sentiment": 'positive', "aspects_positive": [],
         "aspects_negative": [], "language": 'und'},
    24: {"sentiment": 'mixed', "aspects_positive": [],
         "aspects_negative": [], "language": 'und'},
}
for g in GOLD.values():   # tập khía cạnh gộp cả khen lẫn chê
    g["aspects"] = g["aspects_positive"] + g["aspects_negative"]
print("Số đánh giá có nhãn tay:", len(GOLD))

### Phương pháp đối chứng trên 24 đánh giá

Từ khoá chỉ có tiếng Anh, soạn sau khi đọc 24 đánh giá. Từ khoá không phân biệt khen với chê, nên đối chứng không có điểm cho khía cạnh bị chê.

In [ ]:
TU_KHOA_ANH = {
    "location":    r"location|metro|center|walkable",
    "cleanliness": r"clean|dust|dirty|hair|greasy|smell",
    "host":        r"host|check-in",
    "value":       r"value|price|cheap",
    "noise":       r"noisy|noise|loud|quiet",
    "amenities":   r"wifi|bed|kitchen|bathroom",
}

Ô dưới đoán cảm xúc và ngôn ngữ bằng đếm từ, rồi gom dự đoán của cả hai cách theo cùng một dạng.

In [ ]:
# Cảm xúc: đếm từ tốt trừ từ xấu (tiếng Anh). Ngôn ngữ: đếm một số từ đặc trưng của mỗi ngôn ngữ.
TOT_ANH = r"great|perfect|amazing|excellent|lovely|comfy|cozy|spotless|kind|fast|quiet"
XAU_ANH = r"not |dirty|noisy|loud|avoid|cancelled|misleading|smell|hair|greasy|confusing|stranded|tiny"
DAU_HIEU_NGON_NGU = {
    "en": r"\b(the|was|and|but|from)\b",
    "es": r"\b(el|la|muy|pero|hasta|todo)\b",
    "pt": r"\b(o|muito|mas|tempo|bem)\b",
    "fr": r"\b(le|la|est|très|mais)\b",
    "it": r"\b(ottima|molto|appartamento)\b",
}

def dc_khia_canh(text):
    return {a for a, mau in TU_KHOA_ANH.items() if re.search(mau, text, flags=re.I)}

def dc_cam_xuc(text):
    tot = len(re.findall(TOT_ANH, text, flags=re.I))
    xau = len(re.findall(XAU_ANH, text, flags=re.I))
    return "positive" if tot > xau else "negative" if xau > tot else "mixed"

def dc_ngon_ngu(text):
    dem = {lang: len(re.findall(mau, text, flags=re.I)) for lang, mau in DAU_HIEU_NGON_NGU.items()}
    nhieu_nhat = max(dem, key=dem.get)
    return nhieu_nhat if dem[nhieu_nhat] > 0 else "und"     # und: không xác định được

# Gom dự đoán của hai cách theo cùng một dạng: trường -> {mã đánh giá: giá trị}.
cot_van_ban = minh_hoa["comments"]
doi_chung = {
    "sentiment": cot_van_ban.map(dc_cam_xuc).to_dict(),
    "aspects": cot_van_ban.map(dc_khia_canh).to_dict(),
    "language": cot_van_ban.map(dc_ngon_ngu).to_dict(),
}
llm = {
    "sentiment": {ma: r.sentiment for ma, r in hop_le.items()},
    "aspects": {ma: set(r.aspects_positive) | set(r.aspects_negative) for ma, r in hop_le.items()},
    "aspects_negative": {ma: set(r.aspects_negative) for ma, r in hop_le.items()},
    "language": {ma: r.language for ma, r in hop_le.items()},
}
print(pd.DataFrame({"doi_chung": doi_chung["aspects"], "llm": llm["aspects"]}).head(6))

### Chấm điểm bằng nhãn tay

- Cảm xúc, ngôn ngữ: mỗi đánh giá đúng hoặc sai → **tỷ lệ đúng** (accuracy).
- Khía cạnh có nhiều nhãn nên hỏi hai câu. Đánh giá 17: nhãn tay {noise, amenities, host}; LLM thêm location, cleanliness.
  - LLM đưa 5 nhãn, đúng 3 → **precision** = 3/5 = 0,6.
  - Nhãn tay có 3 nhãn, LLM tìm được cả 3 → **recall** = 3/3 = 1.
  - **F1** = 2·P·R / (P + R) = 0,75: chỉ cao khi cả hai số cùng cao.

Bảng điểm bên dưới cộng số nhãn đúng, thừa, thiếu của cả 24 đánh giá rồi mới tính F1; tập khía cạnh gộp cả khen lẫn chê, và có thêm F1 riêng cho khía cạnh bị chê.

### Viết hàm chấm điểm

Hai hàm dùng chung cho phương pháp đối chứng và LLM, nhận bộ nhãn tay làm tham số.

In [ ]:
def ti_le_dung(du_doan, nhan_tay, truong):
    dung = [du_doan[truong].get(ma) == g[truong]
            for ma, g in nhan_tay.items()]
    return sum(dung) / len(dung)

def f1(du_doan, nhan_tay, truong):
    tp = fp = fn = 0
    for ma, g in nhan_tay.items():
        p, t = du_doan[truong].get(ma, set()), set(g[truong])
        tp += len(p & t)
        fp += len(p - t)
        fn += len(t - p)
    return 2 * tp / (2 * tp + fp + fn)

### So phương pháp đối chứng với LLM

24 đánh giá là quá ít để kết luận chắc chắn. Từ khoá đối chứng được soạn theo chính các đánh giá này, nên trên dữ liệu mới điểm đối chứng có thể thấp hơn.

In [ ]:
def cot_diem(d):
    che = (f1(d, GOLD, "aspects_negative")
           if "aspects_negative" in d else np.nan)
    return [ti_le_dung(d, GOLD, "sentiment"), f1(d, GOLD, "aspects"),
            che, ti_le_dung(d, GOLD, "language")]

bang_diem = pd.DataFrame(
    {"doi_chung": cot_diem(doi_chung), "llm": cot_diem(llm)},
    index=["cam_xuc", "khia_canh_f1", "bi_che_f1", "ngon_ngu"])
print(bang_diem.round(2))

### LLM sai ở đâu?

- Đánh giá 3, 7, 10, 21: LLM thêm `amenities` cho nhận xét chung về chỗ ở.
- Đánh giá 13: LLM xếp `location` vào cả hai danh sách; prompt chưa nêu quy ước này.
- Đánh giá 17: LLM thêm hai khía cạnh không có trong văn bản.

Ô dưới so từng kết quả với nhãn tay.

In [ ]:
for ma, g in GOLD.items():
    r = ket_qua.get(ma)
    if r is None:
        print(ma, "không có kết quả hợp lệ")
        continue
    for truong in ["sentiment", "language"]:
        doan = getattr(r, truong)
        if doan != g[truong]:
            print(ma, truong, doan)
    for truong in ["aspects_positive", "aspects_negative"]:
        doan, nhan = set(getattr(r, truong)), set(g[truong])
        if doan != nhan:
            print(ma, truong, "thừa", sorted(doan - nhan),
                  "thiếu", sorted(nhan - doan))

### Hậu kiểm bằng quy tắc

Quy tắc chạy trên mọi kết quả mới, không cần nhãn tay. Schema không chặn được nhãn đúng định dạng nhưng không có căn cứ trong văn bản.

In [ ]:
for ma, r in hop_le.items():
    tu = minh_hoa.loc[ma, "comments"].split()
    so_khia_canh = len(r.aspects_positive) + len(r.aspects_negative)
    hai_chieu = set(r.aspects_positive) & set(r.aspects_negative)
    if not tu and (so_khia_canh > 0 or r.sentiment != "mixed"):
        print(ma, "rỗng nhưng có nhãn:", r.sentiment)
    if hai_chieu:
        print(ma, "vừa khen vừa chê:", sorted(hai_chieu))
    if 0 < len(tu) <= 5 and so_khia_canh >= 2:
        print(ma, f"{len(tu)} từ nhưng có {so_khia_canh} khía cạnh")

### Bài tập 3: Khía cạnh phải có căn cứ

Viết quy tắc: mỗi khía cạnh LLM gán phải khớp ít nhất một từ khoá của khía cạnh đó trong `TU_KHOA_ANH`. In mã đánh giá và khía cạnh không có căn cứ.

1. Quy tắc có bắt được lỗi ở đánh giá 17 không?
2. Những cảnh báo nào là nhầm? Vì sao?

In [ ]:
# Bài tập 3: viết lời giải của bạn ở đây

### Sửa prompt rồi đo lại

1. Đo trên bộ nhãn tay, ghi lại điểm.
2. Đọc các đánh giá bị sai, tìm kiểu lỗi chung.
3. Sửa prompt hoặc schema theo kiểu lỗi, ví dụ thêm quy ước *mỗi khía cạnh chỉ thuộc một danh sách*.
4. Xoá cache, chạy lại cả bộ và đo trên **đúng bộ nhãn tay đó**.

Lưu mỗi phiên bản prompt cùng điểm của nó, như chạy lại test sau mỗi lần sửa code.

### Chi phí tính bằng token

Đo số token trên vài lời gọi, rồi nhân với số đánh giá định xử lý. Đánh giá minh hoạ ngắn hơn đánh giá thật (trung vị 114 ký tự), nên chi phí thật có thể cao hơn. Với hạn mức 500 lời gọi/ngày của gói miễn phí, chạy cả bảng mất gần 4 năm. Vì vậy LLM chỉ chạy trên mẫu.

In [ ]:
vao = sum(v["in_tok"] for v in LLM_CACHE.values()) / len(LLM_CACHE)
ra = sum(v["out_tok"] for v in LLM_CACHE.values()) / len(LLM_CACHE)
print(f"Mỗi đánh giá: {vao:.0f} token vào, {ra:.0f} token ra")
GIA_VAO, GIA_RA = 0.30, 2.50     # USD/1 triệu token, 3.5-flash-lite
for n in [100_000, len(rv)]:
    usd = n * (vao * GIA_VAO + ra * GIA_RA) / 1e6
    print(f"{n:,} đánh giá: khoảng {usd:.0f} USD")

In [ ]:
LOI_GOI_MOI_NGAY = 500           # gói miễn phí, 3.5-flash-lite
for n in [100_000, len(rv)]:
    print(f"{n:,} đánh giá: {n / LOI_GOI_MOI_NGAY:,.0f} ngày ở gói miễn phí")

### Batch API: rẻ hơn, chờ lâu hơn 📖 Tự học

- Gửi một lô nhiều yêu cầu, nhận kết quả sau.
- Giá bằng **50%** giá gọi thường. Google đặt mục tiêu trả kết quả trong 24 giờ; thực tế thường nhanh hơn.
- Hợp với gán nhãn hàng loạt không cần kết quả ngay.

Vẫn cần schema, kiểm tra đầu ra và cache như ở mục 4. Xem [tài liệu Batch API](https://ai.google.dev/gemini-api/docs/batch-api).

### Khi nào dùng cách nào?

| Tình huống | Cách làm |
|---|---|
| Mẫu rõ ràng: mã, giá, ngày | regex (Bài 7) |
| Ngữ nghĩa, nhiều ngôn ngữ, vài nghìn văn bản | LLM + schema + nhãn tay + hậu kiểm |
| Quyết định quan trọng cho từng trường hợp | người thẩm định; LLM chỉ hỗ trợ sàng lọc |

Trước khi dùng nhãn LLM, hãy tự hỏi: nếu 10% nhãn sai, kết luận phân tích có đổi không?

### Làm việc với AI thì sao?

**🤖 AI làm tốt** — viết code gọi API, đề xuất danh mục khía cạnh, sinh đánh giá giả để thử schema.

**⚠️ AI hay sai** — viết theo cách gọi cũ hoặc dùng mô hình đã ngừng hoạt động (như `gemini-2.0-flash`); bỏ qua hạn mức; tự đổi danh mục khía cạnh giữa các lần chạy.

### Kiểm chứng thế nào?

- Code gọi API: đối chiếu **tài liệu chính thức** và tên mô hình hiện hành; chạy thử trên 1–2 đánh giá trước khi chạy hàng loạt.
- Nhãn do LLM gán: đo trên **bộ nhãn tay** và chạy hậu kiểm trước khi dùng trong phân tích.

### Đọc thêm

1. Gemini API: [Interactions](https://ai.google.dev/gemini-api/docs/interactions), [Structured output](https://ai.google.dev/gemini-api/docs/structured-output), [Pricing](https://ai.google.dev/gemini-api/docs/pricing), [Rate limits](https://ai.google.dev/gemini-api/docs/rate-limits)
2. [Pydantic: Models](https://docs.pydantic.dev/latest/concepts/models/)
3. [Bài tập lớn: yêu cầu của hợp phần LLM](https://courses.iaidev.com/programming-for-data-processing/2627-1/projects/project_airbnb.html)

---

## Bài tập về nhà

1. **Sửa prompt rồi đo lại** (cần khoá API; mỗi lần đo là 24 lời gọi).
   1. Lưu bảng điểm hiện tại: `diem_cu = bang_diem.copy()`.
   2. Thêm vào `PROMPT_TEMPLATE` quy ước *mỗi khía cạnh chỉ thuộc một danh sách, theo ý chính của đánh giá*, rồi chạy lại ô đó.
   3. Chạy `LLM_CACHE.clear()`: mọi kết quả trong cache thuộc prompt cũ.
   4. Chạy lại các ô từ `goi_llm` đến bảng điểm (gồm ô gom dự đoán ở mục 5), rồi so với `diem_cu`.
   5. Mô hình không cho cùng kết quả mỗi lần chạy. Chạy prompt cũ thêm một lần để biết điểm dao động bao nhiêu
      trước khi kết luận prompt mới tốt hơn.
2. **Chạy trên đánh giá thật.** Lấy ngẫu nhiên 200 đánh giá dài từ 50 ký tự trong `rv` (`random_state=42`),
   chạy phương pháp đối chứng và LLM, lưu cache ra file JSON. Tự gán nhãn 20 đánh giá theo hướng dẫn gán của bạn,
   tính bảng điểm như mục 5 (truyền bộ nhãn của bạn vào `ti_le_dung`, `f1`) và ước tính chi phí nếu chạy cho cả bảng.
   Nộp notebook kèm file cache để người khác chạy lại được mà không cần khoá API.

In [ ]:
CHAY_BAI_VE_NHA = False   # đổi thành True khi làm bài 2

if CHAY_BAI_VE_NHA:
    danh_gia_dai = rv[rv["comments"].str.len() >= 50]
    mau_that = danh_gia_dai.sample(200, random_state=42)
    print(mau_that.shape)

---

## Tóm tắt bài học

- Văn bản tự do: đổi thành bảng nhãn rồi mới phân tích bằng pandas.
- Làm phương pháp đối chứng trước; đo cả hai cách trên cùng bộ nhãn tay.
- Gọi LLM qua API với schema, tôn trọng hạn mức và dùng cache; lưu cache ra file để chạy lại không cần khoá.
- Đúng schema chưa chắc đúng nội dung: cần nhãn tay và quy tắc hậu kiểm; chi phí tính bằng token.

Hoàn thành ba bài tập rồi chạy **Restart session and run all**.

**Khai báo AI (nếu dùng):** công cụ, yêu cầu chính, một kết quả bạn đã tự kiểm tra. …

**Bài sau:** trực quan hoá dữ liệu cơ bản với matplotlib và pandas.